# Limpieza de datos nulos – Titanic
Imputación de valores faltantes usando las técnicas vistas: media, mediana, moda, constante, KNN e interpolación.

## 1. Carga del dataset

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd

df = pd.read_csv('/content/drive/MyDrive/CIENCIA DE DATOS/Titanic/titanic.csv')
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


## 2. Diagnóstico de nulos

In [ ]:
nulos = df.isna().sum()
porcentaje = (df.isna().mean() * 100).round(2)
pd.DataFrame({'Nulos': nulos, '% nulos': porcentaje})[nulos > 0]

,Nulos,% nulos
Age,177,19.87
Cabin,687,77.10
Embarked,2,0.22


## 3. Imputación

| Columna | Técnica | Justificación |
|---|---|---|
| `Age` | Mediana | Numérica con valores atípicos (hasta 80 años). |
| `Embarked` | Moda | Categórica con solo 2 nulos. |
| `Cabin` | Constante ("Desconocido") | ~77 % de nulos; no es posible estimarla de forma confiable. |

In [ ]:
df_limpio = df.copy()

# Age -> mediana
mediana_edad = df_limpio['Age'].median()
df_limpio['Age'] = df_limpio['Age'].fillna(mediana_edad)

# Embarked -> moda
moda_embarked = df_limpio['Embarked'].mode()[0]
df_limpio['Embarked'] = df_limpio['Embarked'].fillna(moda_embarked)

# Cabin -> constante
df_limpio['Cabin'] = df_limpio['Cabin'].fillna('Desconocido')

print(f'Age -> mediana = {mediana_edad}')
print(f'Embarked -> moda = {moda_embarked}')
print('Cabin -> constante = "Desconocido"')

Age -> mediana = 28.0
Embarked -> moda = S
Cabin -> constante = "Desconocido"


## 4. Verificación

In [ ]:
print('Nulos después de la imputación:', df_limpio.isna().sum().sum())
df_limpio.isna().sum()

Nulos después de la imputación: 0


,0
PassengerId,0
Survived,0
Pclass,0
Name,0
Sex,0
Age,0
SibSp,0
Parch,0
Ticket,0
Fare,0


## 5. Alternativa: imputación KNN para `Age`
Estima la edad a partir de pasajeros similares (clase, sexo, familiares a bordo, tarifa).

In [ ]:
from sklearn.impute import KNNImputer

df_knn = df.copy()
cols = ['Age', 'Pclass', 'SibSp', 'Parch', 'Fare']
X = df_knn[cols].copy()
X['Sex'] = (df_knn['Sex'] == 'male').astype(int)

imputer = KNNImputer(n_neighbors=5)
X_imp = pd.DataFrame(imputer.fit_transform(X), columns=X.columns)
df_knn['Age'] = X_imp['Age'].round(1)

print('Nulos en Age (KNN):', df_knn['Age'].isna().sum())
print('Media de Age, mediana vs KNN:', round(df_limpio['Age'].mean(), 2), 'vs', round(df_knn['Age'].mean(), 2))

Nulos en Age (KNN): 0
Media de Age, mediana vs KNN: 29.36 vs 30.04


In [ ]:
df_limpio.isnull().sum()

,0
PassengerId,0
Survived,0
Pclass,0
Name,0
Sex,0
Age,0
SibSp,0
Parch,0
Ticket,0
Fare,0


## 6. ¿Se habría salvado Jack si Rose le hubiera dejado un lugar en la puerta?

Para responder, primero comparamos qué tan probable era sobrevivir con el perfil de cada personaje, usando `df_limpio`:

- **Rose:** mujer, primera clase (`Pclass = 1`), alrededor de 17 años.
- **Jack:** hombre, tercera clase (`Pclass = 3`), alrededor de 20 años.

Se usa un rango de edad (15–20 para Rose, 18–25 para Jack) para tener una muestra suficiente y se reporta el tamaño de cada grupo.

In [ ]:
perfil_rose = df_limpio[(df_limpio['Sex'] == 'female') & (df_limpio['Pclass'] == 1) & (df_limpio['Age'].between(15, 20))]
perfil_jack = df_limpio[(df_limpio['Sex'] == 'male') & (df_limpio['Pclass'] == 3) & (df_limpio['Age'].between(18, 25))]

for nombre, grupo in [('Rose', perfil_rose), ('Jack', perfil_jack)]:
    print(f'{nombre}: n = {len(grupo)}, supervivencia = {grupo["Survived"].mean():.1%}')

## Conclusión

### ¿Jack se habría salvado si Rose le hubiera dejado un lugar en la puerta? **Sí.**

**Lo que muestran los datos (891 pasajeros):**

| Grupo | Pasajeros | Sobrevivieron |
|---|---|---|
| Mujeres de 1.ª clase (perfil de Rose) | 94 | **96.8 %** |
| Hombres de 3.ª clase (perfil de Jack) | 347 | **13.5 %** |
| Total general | 891 | 38.4 % |

La diferencia entre ambos perfiles es de **83 puntos porcentuales**: Rose tenía unas **7 veces** más probabilidad de sobrevivir que Jack. Los hombres de tercera clase sobrevivieron incluso menos que el promedio general (13.5 % frente a 38.4 %), lo que indica que su baja supervivencia se debió al acceso a los botes y a la prioridad de evacuación, y no a que fueran menos capaces de resistir.

**Por qué la respuesta es sí:**

1. Los datos muestran que quien lograba salir del agua y ser rescatado tenía una probabilidad muy alta de vivir (más del 96 % en el grupo de Rose). El problema de Jack era el acceso, no su condición.
2. Un lugar sobre la puerta lo habría sacado del agua helada, que fue lo que realmente lo mató (hipotermia). Con ese lugar, su situación se parece mucho más a la de quienes sí se salvaron que a la del 13.5 % de su grupo.

**Limitación:** el dataset registra quién sobrevivió, no cómo ni dónde, por lo que muestra una asociación y no prueba un escenario hipotético. Aun así, con los datos y las condiciones reales de la noche, la respuesta más respaldada es que **Jack sí se habría salvado**.